> 📄 English: [fine_tuning_agents_guide.ipynb](fine_tuning_agents_guide.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--fine-tunning-agents--fine-tuning-agents-guide)

# 通过微调增强 AI Agent 指南
## 概览
本教程演示如何利用监督微调（SFT）创建更强大、更高效的 AI Agent。虽然我们以银行场景为例进行演示，但这里介绍的技术和原则适用于任何需要通过微调来增强 AI Agent 能力的场景。

## 什么是微调（Fine-Tuning）？

微调是指取一个预训练 AI 模型——一个已经从海量数据中学习到通用语言模式的模型——再在更小、更专门的数据集上进一步训练它的过程。

这种额外训练帮助模型适应特定领域、任务或沟通风格，使其对目标应用更有效、更可靠。微调让你能够自定义模型的行为、词汇和输出格式，以更好地满足你的独特需求。

## 为什么要微调 AI Agent？
微调为 AI Agent 开发带来了若干强大优势：

1. **领域专长**：
   - 内化特定领域的法规与合规要求

   - 减少专门场景下的幻觉

2. **一致的风格与语气**：
   - 学会持续保持你的品牌声音

   - 在所有交互中提供统一的响应

3. **结构化输出**：
   - 可靠地生成特定格式（JSON、SQL、Markdown）

4. **减少提示词工程**：
   - 无需冗长的 system 提示词

   - 显著减少 token 使用量和成本

5. **提升效率**：
   - 更小的微调模型可以替代更大的基座模型

   - 专门任务响应更快


### 谁应该考虑微调？
- 为特定领域构建专门 AI Agent 的团队

- 响应速度和成本很重要的应用

- 处理复杂领域专属任务的系统

## 方法概览
1. **数据准备**：创建能捕获期望行为的高质量训练数据集

2. **监督微调**：让基座模型适应特定需求

3. **集成**：用微调模型构建高效的 Agent 工作流

4. **评估**：衡量各项关键指标的性能改进


这种方法为通过微调增强 AI Agent 提供了稳健的框架，使其能更有效地执行专门任务，同时保持效率和可靠性。

## 结论
本教程展示了如何利用微调创建更强大的 AI Agent。无论你需要领域专长、一致的格式还是更高的效率，微调都能帮助你的 Agent 表现得更好，同时降低成本和复杂度。这里演示的技术可应用于任何需要专门 Agent 行为的行业或用例。

作者：[Zohar Mosseri](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--fine-tunning-agents--fine-tuning-agents-guide&click=linkedin-zohar-mosseri&target=https%3A%2F%2Fwww.linkedin.com%2Fin%2Fzohar-mosseri%2F&text=Zohar%20Mosseri)


📖 **如需了解更多关于微调 AI 模型及其应用的背景知识，请阅读我们的详细博客文章：** [Fine-tuning AI models](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--fine-tunning-agents--fine-tuning-agents-guide&click=substack-diamantai&target=https%3A%2F%2Fdiamantai.substack.com%2Fp%2Ffine-tuning-ai-models-how-they-evolve&text=Fine-tuning%20AI%20models)

*DiamantAI 是一份位居前 0.1% 的 AI 通讯，拥有超过 25,000 名订阅者，专注于 AI 技术、突破和教程。*

---


## 示例用例：银行助手
为了演示这些概念，我们将创建一个银行客户支持助手。这个示例将展示如何：
- 为领域专属知识组织训练数据
- 针对专门任务微调模型
- 将微调后的模型集成到实际应用中


## 1. 配置与导入

安装并导入必要的库


In [ ]:
%pip install --upgrade openai langchain langgraph==0.3.1 tiktoken pandas scikit-learn

import os, json, time
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables and set OpenAI API key
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

# Initialize OpenAI client
client = OpenAI()

## 2. 为微调准备训练与验证文件

OpenAI 期望微调使用 **UTF-8 编码的 JSONL** 文件，每行一个 JSON 对象。
每个示例都应遵循包含 system、user 和 assistant 消息的聊天格式。

### 理解训练数据格式
微调模型的质量在很大程度上取决于你的训练数据。以下是你需要知道的：

1. **文件格式**：JSONL（JSON Lines），UTF-8 编码

2. **消息结构**：每个示例必须包含带角色和内容的消息序列

3. **必需字段**：每条消息都需要 `role` 和 `content` 字段

4. **有效角色**：`system`、`user` 和 `assistant`

### 示例格式
```jsonl
{
  "messages": [
    {"role": "system", "content": "You are a helpful domain expert assistant."},
    {"role": "user", "content": "What is the policy on this matter?"},
    {"role": "assistant", "content": "According to our policies..."}
  ]
}
```

### 创建训练数据：一个银行示例
为了演示这些原则，让我们使用一个银行知识库来创建训练数据集。这个示例可以适用于任何领域专属知识：


In [ ]:
import json
import random
import pathlib
import pandas as pd


# Create directory for our fine-tuning data
DATA_DIR = pathlib.Path("bank_finetune_data")
DATA_DIR.mkdir(exist_ok=True)

# --- General Approach to Creating Training Data ---
# 1. Use your domain knowledge base (docs, FAQs, policies, etc.)
# 2. Structure content into clear topics
# 3. Create diverse, well-phrased Q&A pairs

# Example using banking domain
kb_docs = [
    {
        "title": "Account Types",
        "content": {
            "checking": "We offer three types of checking accounts: Basic (no minimum balance), Premium ($2,500 minimum, no fees), and Student (no fees with valid student ID).",
            "savings": "Our savings accounts include Regular (0.5% APY), High-Yield (1.5% APY with $10,000 minimum), and Goal-Based savings with customizable targets.",
            "business": "Business accounts feature unlimited transactions, merchant services integration, and dedicated support. Available in Standard and Premium tiers."
        }
    },
    # Additional examples below - uncomment to create a more comprehensive training set
    # For tutorial purposes, we'll start with a single example to demonstrate the process
    # {
    #     "title": "Security Protocols",
    #     "content": {
    #         "authentication": "We use multi-factor authentication, biometric verification, and device recognition. Login attempts are limited to 3 tries before temporary lockout.",
    #         "fraud_prevention": "Real-time transaction monitoring, instant fraud alerts, and zero liability protection for unauthorized charges.",
    #         "data_protection": "End-to-end encryption for all transactions, secure socket layer (SSL) protection, and regular security audits."
    #     }
    # },
    # {
    #     "title": "Digital Banking",
    #     "content": {
    #         "mobile_features": "Mobile check deposit, peer-to-peer payments, bill pay, and account aggregation across institutions.",
    #         "online_services": "Budget tracking, spending analysis, electronic statements, and customizable alerts.",
    #         "technical_requirements": "Compatible with iOS 13+ and Android 8+. Requires secure internet connection and updated browser versions."
    #     }
    # }
]

def generate_diverse_examples(kb_doc):
    """
    Generate multiple training examples with diverse phrasings and scenarios.
    Following best practices for high-quality training data.
    """
    examples = []
    content = kb_doc["content"]
    
    # Generate examples for each subtopic
    for subtopic, details in content.items():
        # Multiple question variations
        questions = [
            f"Can you explain {subtopic}?",
            f"What should I know about {subtopic}?",
            f"Tell me about your {subtopic}",
            f"How does {subtopic} work?",
            f"I need information regarding {subtopic}",
            f"What are the details of {subtopic}?"
        ]
        
        # Multiple response variations with consistent style
        responses = [
            f"Here's what you need to know about {subtopic}: {details}",
            f"Regarding {subtopic}: {details} Let me know if you need any clarification.",
            f"I'll explain {subtopic}. {details} Is there anything specific you'd like to know more about?",
            f"{details} This information about {subtopic} is current as of today. Please check our website for any updates."
        ]
        
        # Generate combinations with system message variations
        system_messages = [
            "You are a knowledgeable banking assistant focused on providing accurate, compliant information.",
            "You are a helpful financial services expert committed to clear, precise communication.",
            "You are a banking specialist dedicated to providing detailed, accurate responses."
        ]
        
        # Create multiple examples for each subtopic
        for system_msg in system_messages:
            for question in questions:
                for response in responses:
                    examples.append({
                        "messages": [
                            {"role": "system", "content": system_msg},
                            {"role": "user", "content": question},
                            {"role": "assistant", "content": response}
                        ]
                    })
    
    return examples

# Generate comprehensive training dataset
examples = []
for doc in kb_docs:
    examples.extend(generate_diverse_examples(doc))

# Shuffle examples to ensure good distribution
random.shuffle(examples)

print(f"Generated {len(examples)} diverse training examples")


### 微调的数据质量最佳实践

微调模型的成功取决于结构良好的训练数据。关键原则：

1. **数据要求**
   - 每种行为至少 10 个示例
   - 平衡各类别，并保留 10–20% 用于验证

2. **质量准则**
   - 每个示例中意图和响应要清晰
   - 变化措辞，保持格式和风格一致
   - 反映真实使用场景

3. **常见陷阱**
   - 避免敏感信息、风格不一致、重复以及过拟合

记住：质量重于数量——一小撮高质量示例胜过一大堆糟糕的示例。


### 划分训练集与验证集

微调中至关重要的一步是把数据集划分为训练集和验证集。训练集用来教模型，而验证集用来衡量模型在未见数据上的表现。我们将使用 80/20 的划分比例——这是一个常见的比例，既能提供足够的验证数据，又能最大化训练示例。


In [ ]:
# Randomly shuffle examples to ensure even distribution between sets
random.shuffle(examples)

# Calculate split point for 80% training, 20% validation
split_index = int(0.8 * len(examples))

# Split examples into training and validation sets
train_examples, validation_examples = examples[:split_index], examples[split_index:]

# Save to JSONL files
train_file = DATA_DIR / "train.jsonl"
validation_file = DATA_DIR / "validation.jsonl"

with open(train_file, "w", encoding="utf-8") as f:
    for example in train_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

with open(validation_file, "w", encoding="utf-8") as f:
    for example in validation_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print(f"✍️ Created {len(train_examples)} training examples and {len(validation_examples)} validation examples")

# Create a DataFrame for easy visualization of example structure
df = pd.DataFrame([
    {
        "system": ex["messages"][0]["content"],
        "user": ex["messages"][1]["content"],
        "assistant": ex["messages"][2]["content"]
    }
    for ex in train_examples[:3]
])
# Display first 3 examples showing system, user, and assistant messages
df

### 校验 JSONL 格式

在上传之前，让我们校验文件，以确保它们满足 OpenAI 的要求：


In [ ]:
def validate_jsonl(file_path):
    """Validate that the file is proper JSONL format for fine-tuning."""
    with open(file_path, 'r', encoding='utf-8') as f:
        line_count = 0
        for line in f:
            line_count += 1
            try:
                data = json.loads(line)
                # Check if it has the messages field
                if 'messages' not in data:
                    return False, f"Line {line_count} missing 'messages' field"
                # Check message format
                for msg in data['messages']:
                    if 'role' not in msg or 'content' not in msg:
                        return False, f"Line {line_count} has message missing 'role' or 'content'"
                    if msg['role'] not in ['system', 'user', 'assistant']:
                        return False, f"Line {line_count} has invalid role: {msg['role']}"
            except json.JSONDecodeError:
                return False, f"Line {line_count} is not valid JSON"
    return True, f"Validated {line_count} examples"

# Validate our files
train_valid, train_msg = validate_jsonl(train_file)
val_valid, val_msg = validate_jsonl(validation_file)

print(f"Training file: {'✅' if train_valid else '❌'} {train_msg}")
print(f"Validation file: {'✅' if val_valid else '❌'} {val_msg}")

## 3. 将 JSONL 文件上传到 OpenAI 并启动微调

我们将使用 OpenAI 的 Python 客户端以编程方式上传文件：


In [ ]:
import openai
import os
from pathlib import Path

# Initialize OpenAI client
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def upload(path):
    """Upload a file to OpenAI for fine-tuning"""
    with open(path, "rb") as file:
        resp = client.files.create(
            file=file,
            purpose="fine-tune"  # Specify the file will be used for fine-tuning
        )
    print(f"Uploaded {Path(path).name} → {resp.id}")
    return resp.id

# Upload our training and validation files
# These IDs will be used when creating the fine-tuning job
train_file_id = upload(train_file)
validation_file_id = upload(validation_file)

你也可以通过 UI 上传：

1. 导航到 dashboard > [fine-tuning](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--fine-tunning-agents--fine-tuning-agents-guide&click=openai-platform&target=https%3A%2F%2Fplatform.openai.com%2Ffinetune&text=fine-tuning)。

2. 点击 + Create。

3. 在 Training data 下上传你的 JSONL 文件。


## 4. 启动微调任务
### 使用基座模型和 JSONL 文件创建微调任务


In [ ]:
job = client.fine_tuning.jobs.create(
    # The ID of the JSONL file containing training examples
    training_file=train_file_id,
    # The ID of the JSONL file containing validation examples (optional but recommended)
    validation_file=validation_file_id,
    model="gpt-4o-mini-2024-07-18",  # base model
    suffix="banking-support"  # custom suffix for the fine-tuned model
)
# Print job information for monitoring
print(f"Fine-tuning job created with ID: {job.id}")
print(f"Job status: {job.status}")
print(f"Monitor progress at: https://platform.openai.com/finetune")

## 5. 监控任务状态


In [ ]:
import time

# Continuously check job status every 30 seconds
while True:
    # Retrieve current status of the fine-tuning job
    status = client.fine_tuning.jobs.retrieve(job.id).status
    print("Status:", status)
    if status in ("succeeded", "failed", "cancelled"):
        break      # Exit loop if job reaches a terminal state
    # Wait 30 seconds before next status check
    time.sleep(30)

# Once complete, get the identifier for the fine-tuned model
fine_tuned_model = openai.fine_tuning.jobs.retrieve(job.id).fine_tuned_model  # e.g. 'ft:gpt‑4o-mini:banking-support:2025-05-12-10-30-02'
print("✅ Fine‑tuned model:", fine_tuned_model)

### 理解微调指标

你可以在 OpenAI 仪表盘的 [fine-tuning](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--fine-tunning-agents--fine-tuning-agents-guide&click=openai-platform&target=https%3A%2F%2Fplatform.openai.com%2Ffinetune&text=fine-tuning) 标签页中查看进度和损失曲线。根据数据集大小和模型复杂度，微调过程通常需要几分钟到几个小时。


微调时，你会看到两个关键图表：

#### 损失曲线（Loss Curves）
损失曲线展示模型随时间的学习情况：

- **训练损失**（🟢 绿色）：模型在训练数据上的误差（应该下降）

- **验证损失**（🟣 紫色）：在未见数据上的误差（应跟随训练损失）

- 如果验证损失上升而训练损失下降 = 过拟合

#### 准确率曲线（Accuracy Curves）
这些曲线展示模型随时间变化的预测准确率：

- **训练准确率**（🟢 绿色）：应稳步上升

- **验证准确率**（🟣 紫色）：应跟随训练曲线
- 如果它们背离 = 可能过拟合

#### 关注点
✅ **良好迹象**
- 两个指标都稳步改善

- 训练和验证曲线呈相似趋势

⚠️ **警告迹象**
- 验证损失上升而训练损失下降

- 训练与验证指标之间存在较大差距

- 曲线起伏不定或不稳定

![Fine-tuning metrics showing loss and accuracy curves](https://i.ibb.co/hx55yKKH/loss-accuracy-curve.png)


## 6. 实现你的微调模型

本节演示如何使用 LangGraph 将你的微调模型集成到生产环境中。你将学习如何：
- 将你的专门模型连接到现有工具和系统

- 组织交互流程

- 有效处理领域专属查询


In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent


# Define example tool to demonstrate fine-tuned model integration
@tool
def account_lookup(account_id: str) -> str:
    """Look up account information by account ID (requires authentication)."""
    return f"Account {account_id} information is available after authentication."

@tool
def transfer_payment(source_account: str, destination_account: str, amount: float) -> str:
    """Transfer payment from one account to another."""
    return f"Payment of {amount} from {source_account} to {destination_account} has been initiated."


# Use our fine-tuned model
domain_expert = ChatOpenAI(model=fine_tuned_model, temperature=0.0)

# Create the agent with our tools
agent = create_react_agent(
    model=domain_expert,  # Use our fine-tuned model
    tools=[account_lookup, transfer_payment]
    )

# Test the agent with some example queries
test_queries = [
    "What account types do you offer?",
    "What are the fees for wire transfers?",
    "I need information about overdraft protection"
]

# Run the agent on a test query
for query in test_queries:
    response = agent.invoke({"messages": [HumanMessage(content=query)]})
    print("\nQuery:", query)
    print("\nResponse:")
    # Extract and print only the final assistant response
    for msg in reversed(response["messages"]):
        # Check if it's an AIMessage and has non-empty content
        if hasattr(msg, "content") and msg.content and msg.__class__.__name__ == "AIMessage":
            print(msg.content)
            break

*提示：如果之后你需要多个专门的模型，只需让每个 LangGraph 节点指向不同的微调模型即可。*


## 7. 微调的优势：对比分析

让我们在两个关键方面比较微调模型与基座 LLM 的表现：

### 1. 领域专长与安全
**查询：** "I want to transfer $50,000 to account 1234567"

**基座 LLM：**
> 为了您的安全，请确保您使用的是安全且经过验证的银行平台来执行此操作。如果您愿意，我可以帮您起草一份示例确认消息，或说明通过您的银行或应用转账的常见步骤。您希望我这样做吗？

**微调模型：**
> 为了您的安全，我需要：
> 1. 通过我们的身份验证流程验证您的身份
> 2. 确认收款账户详情
> 3. 检查这笔转账是否需要额外的反欺诈措施（超过 $10,000 的转账需要）
>
> 请提供您的账户 ID，以开始安全的验证流程。

### 2. 结构化响应与产品知识
**查询：** "What savings accounts do you offer?"

**基座 LLM：**
> 银行通常会提供各种类型的储蓄账户，具有不同的利率和最低余额要求。您应该向您的具体银行咨询其提供的产品。

**微调模型：**
```json
{
    "savings_products": {
        "regular_savings": {
            "rate": "0.5% APY",
            "min_balance": "$0",
            "features": ["Mobile banking", "24/7 support"]
        },
        "high_yield_savings": {
            "rate": "1.5% APY",
            "min_balance": "$10,000",
            "features": ["Premium interest rate", "Priority support"]
        }
    }
}
```

**结论：**

这些改进仅靠提示词工程是难以实现的，这体现了微调在创建专门 AI Agent 方面的价值。


## 8. 微调最佳实践

| 主题                   | 建议                                                                                   |
|-------------------------|--------------------------------------------------------------------------------------------------|
| **训练/验证划分** | 始终保留至少 10% 的数据（理想情况 ≥40 个示例）用于验证。这有助于你监控过拟合，并确保模型对新查询有良好的泛化能力。 |
| **早停（Early Stopping）**         | 默认启用。如果验证损失停止改善，训练会自动停止，从而防止过拟合并节省算力成本。 |
| **示例截断**     | 任何长于模型上下文窗口的样本都会被自动截断。为获得最佳效果，请保持消息简洁且相关。 |
| **评估指标**     | OpenAI 仪表盘显示 token 级别的准确率。如需更深入的分析，可以考虑在你自己的测试集上运行 BLEU 或精确匹配（exact-match）指标。 |


## 9. 下一步

- 自动化数据刷新：
  定期（例如每晚）用新的领域数据重新训练，使模型跟上不断变化的知识和需求。

- 添加合规与审计节点：
  集成额外的 LangGraph 节点（可能带有它们各自的微调模型），以自动检查响应的合规性并标记敏感主题。

- 使用 LangSmith 等工具监控 Agent 的性能和成本，并持续评估真实用户查询，以迭代改进你的训练数据和工作流。
